https://mlu-explain.github.io/logistic-regression/

In [1]:
import random
import math


def sigmoid(z):
    z = max(-500, min(500, z))
    return 1.0 / (1.0 + math.exp(-z))


random.seed(42)
N = 200
X = []
y = []

for _ in range(N // 2):
    X.append([random.gauss(2, 1), random.gauss(2, 1)])
    y.append(0)

for _ in range(N // 2):
    X.append([random.gauss(5, 1), random.gauss(5, 1)])
    y.append(1)

combined = list(zip(X, y))
random.shuffle(combined)
X, y = zip(*combined)
X = list(X)
y = list(y)

print(f"Generated {N} samples (2 classes, 2 features)")
print(f"Class 0 center: (2, 2), Class 1 center: (5, 5)")
print(f"First 5 samples:")
for i in range(5):
    print(f"  Features: [{X[i][0]:.2f}, {X[i][1]:.2f}], Label: {y[i]}")


class LogisticRegression:
    def __init__(self, n_features, learning_rate=0.01):
        self.weights = [0.0] * n_features
        self.bias = 0.0
        self.lr = learning_rate
        self.loss_history = []

    def predict_proba(self, x):
        z = sum(w * xi for w, xi in zip(self.weights, x)) + self.bias
        return sigmoid(z)

    def predict(self, x, threshold=0.5):
        return 1 if self.predict_proba(x) >= threshold else 0

    def compute_loss(self, X, y):
        n = len(y)
        total = 0.0
        for i in range(n):
            p = self.predict_proba(X[i])
            p = max(1e-15, min(1 - 1e-15, p))
            total += y[i] * math.log(p) + (1 - y[i]) * math.log(1 - p)
        return -total / n

    def fit(self, X, y, epochs=1000, print_every=200):
        n = len(y)
        n_features = len(X[0])
        for epoch in range(epochs):
            dw = [0.0] * n_features
            db = 0.0
            for i in range(n):
                p = self.predict_proba(X[i])
                error = p - y[i]
                for j in range(n_features):
                    dw[j] += error * X[i][j]
                db += error
            for j in range(n_features):
                self.weights[j] -= self.lr * (dw[j] / n)
            self.bias -= self.lr * (db / n)
            loss = self.compute_loss(X, y)
            self.loss_history.append(loss)
            if epoch % print_every == 0:
                print(f"  Epoch {epoch:4d} | Loss: {loss:.4f} | w: [{self.weights[0]:.3f}, {self.weights[1]:.3f}] | b: {self.bias:.3f}")
        return self

    def accuracy(self, X, y):
        correct = sum(1 for i in range(len(y)) if self.predict(X[i]) == y[i])
        return correct / len(y)


split = int(0.8 * N)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print("\n=== Training Logistic Regression ===")
model = LogisticRegression(n_features=2, learning_rate=0.1)
model.fit(X_train, y_train, epochs=1000, print_every=200)

print(f"\nTrain accuracy: {model.accuracy(X_train, y_train):.4f}")
print(f"Test accuracy:  {model.accuracy(X_test, y_test):.4f}")
print(f"Weights: [{model.weights[0]:.4f}, {model.weights[1]:.4f}]")
print(f"Bias: {model.bias:.4f}")


Generated 200 samples (2 classes, 2 features)
Class 0 center: (2, 2), Class 1 center: (5, 5)
First 5 samples:
  Features: [5.67, 3.86], Label: 1
  Features: [0.88, 2.46], Label: 0
  Features: [1.53, 2.50], Label: 0
  Features: [4.60, 6.20], Label: 1
  Features: [6.30, 6.90], Label: 1

=== Training Logistic Regression ===
  Epoch    0 | Loss: 0.6289 | w: [0.074, 0.070] | b: -0.001
  Epoch  200 | Loss: 0.3136 | w: [0.464, 0.333] | b: -2.402
  Epoch  400 | Loss: 0.2173 | w: [0.657, 0.497] | b: -3.756
  Epoch  600 | Loss: 0.1723 | w: [0.795, 0.608] | b: -4.684
  Epoch  800 | Loss: 0.1462 | w: [0.902, 0.693] | b: -5.391

Train accuracy: 0.9875
Test accuracy:  1.0000
Weights: [0.9886, 0.7629]
Bias: -5.9623


In [3]:
class ClassificationMetrics:
    def __init__(self, y_true, y_pred):
        self.tp = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 1)
        self.tn = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 0)
        self.fp = sum(1 for t, p in zip(y_true, y_pred) if t == 0 and p == 1)
        self.fn = sum(1 for t, p in zip(y_true, y_pred) if t == 1 and p == 0)

    def accuracy(self):
        total = self.tp + self.tn + self.fp + self.fn
        return (self.tp + self.tn) / total if total > 0 else 0

    def precision(self):
        denom = self.tp + self.fp
        return self.tp / denom if denom > 0 else 0

    def recall(self):
        denom = self.tp + self.fn
        return self.tp / denom if denom > 0 else 0

    def f1(self):
        p = self.precision()
        r = self.recall()
        return 2 * p * r / (p + r) if (p + r) > 0 else 0

    def print_confusion_matrix(self):
        print(f"\n  Confusion Matrix:")
        print(f"                  Predicted")
        print(f"                  Pos   Neg")
        print(f"  Actual Pos     {self.tp:4d}  {self.fn:4d}")
        print(f"  Actual Neg     {self.fp:4d}  {self.tn:4d}")

    def print_report(self):
        self.print_confusion_matrix()
        print(f"\n  Accuracy:  {self.accuracy():.4f}")
        print(f"  Precision: {self.precision():.4f}")
        print(f"  Recall:    {self.recall():.4f}")
        print(f"  F1 Score:  {self.f1():.4f}")


y_pred_test = [model.predict(x) for x in X_test]
print("\n=== Classification Report (Test Set) ===")
metrics = ClassificationMetrics(y_test, y_pred_test)
metrics.print_report()





=== Classification Report (Test Set) ===

  Confusion Matrix:
                  Predicted
                  Pos   Neg
  Actual Pos       21     0
  Actual Neg        0    19

  Accuracy:  1.0000
  Precision: 1.0000
  Recall:    1.0000
  F1 Score:  1.0000


Decision boundary

In [4]:
print("\n=== Decision Boundary ===")
w1, w2 = model.weights
b = model.bias
print(f"Decision boundary: {w1:.4f}*x1 + {w2:.4f}*x2 + {b:.4f} = 0")
if abs(w2) > 1e-10:
    print(f"Solved for x2:     x2 = {-w1/w2:.4f}*x1 + {-b/w2:.4f}")

print("\nSample predictions near the boundary:")
test_points = [
    [3.0, 3.0],
    [3.5, 3.5],
    [4.0, 4.0],
    [2.5, 2.5],
    [5.0, 5.0],
]
for point in test_points:
    prob = model.predict_proba(point)
    pred = model.predict(point)
    print(f"  [{point[0]}, {point[1]}] -> prob={prob:.4f}, class={pred}")



=== Decision Boundary ===
Decision boundary: 0.9886*x1 + 0.7629*x2 + -5.9623 = 0
Solved for x2:     x2 = -1.2958*x1 + 7.8152

Sample predictions near the boundary:
  [3.0, 3.0] -> prob=0.3301, class=0
  [3.5, 3.5] -> prob=0.5419, class=1
  [4.0, 4.0] -> prob=0.7396, class=1
  [2.5, 2.5] -> prob=0.1703, class=0
  [5.0, 5.0] -> prob=0.9424, class=1


SoftMaxRegression

In [7]:
class SoftmaxRegression:
    def __init__(self, n_features, n_classes, learning_rate=0.01):
        self.n_features = n_features
        self.n_classes = n_classes
        self.lr = learning_rate
        self.weights = [[0.0] * n_features for _ in range(n_classes)]
        self.biases = [0.0] * n_classes

    def softmax(self, scores):
        max_score = max(scores)
        exp_scores = [math.exp(s - max_score) for s in scores]
        total = sum(exp_scores)
        return [e / total for e in exp_scores]

    def predict_proba(self, x):
        scores = [
            sum(self.weights[k][j] * x[j] for j in range(self.n_features)) + self.biases[k]
            for k in range(self.n_classes)
        ]
        return self.softmax(scores)

    def predict(self, x):
        probs = self.predict_proba(x)
        return probs.index(max(probs))

    def fit(self, X, y, epochs=1000, print_every=200):
        n = len(y)
        for epoch in range(epochs):
            grad_w = [[0.0] * self.n_features for _ in range(self.n_classes)]
            grad_b = [0.0] * self.n_classes
            total_loss = 0.0
            for i in range(n):
                probs = self.predict_proba(X[i])
                for k in range(self.n_classes):
                    target = 1.0 if y[i] == k else 0.0
                    error = probs[k] - target
                    for j in range(self.n_features):
                        grad_w[k][j] += error * X[i][j]
                    grad_b[k] += error
                true_prob = max(probs[y[i]], 1e-15)
                total_loss -= math.log(true_prob)
            for k in range(self.n_classes):
                for j in range(self.n_features):
                    self.weights[k][j] -= self.lr * (grad_w[k][j] / n)
                self.biases[k] -= self.lr * (grad_b[k] / n)
            if epoch % print_every == 0:
                print(f"  Epoch {epoch:4d} | Loss: {total_loss / n:.4f}")
        return self

    def accuracy(self, X, y):
        correct = sum(1 for i in range(len(y)) if self.predict(X[i]) == y[i])
        return correct / len(y)


random.seed(42)
X_3class = []
y_3class = []

centers = [(1, 1), (5, 1), (3, 5)]
for label, (cx, cy) in enumerate(centers):
    for _ in range(50):
        X_3class.append([random.gauss(cx, 0.8), random.gauss(cy, 0.8)])
        y_3class.append(label)

combined = list(zip(X_3class, y_3class))
random.shuffle(combined)
X_3class, y_3class = zip(*combined)
X_3class = list(X_3class)
y_3class = list(y_3class)

split_3 = int(0.8 * len(X_3class))
X_train_3 = X_3class[:split_3]
y_train_3 = y_3class[:split_3]
X_test_3 = X_3class[split_3:]
y_test_3 = y_3class[split_3:]

print("\n=== Multi-class Softmax Regression (3 classes) ===")
softmax_model = SoftmaxRegression(n_features=2, n_classes=3, learning_rate=0.1)
softmax_model.fit(X_train_3, y_train_3, epochs=1000, print_every=200)
print(f"\nTrain accuracy: {softmax_model.accuracy(X_train_3, y_train_3):.4f}")
print(f"Test accuracy:  {softmax_model.accuracy(X_test_3, y_test_3):.4f}")

print("\nSample predictions:")
for i in range(5):
    probs = softmax_model.predict_proba(X_test_3[i])
    pred = softmax_model.predict(X_test_3[i])
    print(f"  True: {y_test_3[i]}, Predicted: {pred}, Probs: [{', '.join(f'{p:.3f}' for p in probs)}]")


print("\n=== Threshold Tuning ===")
print("Default threshold: 0.5. Adjusting trades precision for recall.\n")

thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]
print(f"{'Threshold':>10} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 52)

for t in thresholds:
    y_pred_t = [1 if model.predict_proba(x) >= t else 0 for x in X_test]
    m = ClassificationMetrics(y_test, y_pred_t)
    print(f"{t:>10.1f} {m.accuracy():>10.4f} {m.precision():>10.4f} {m.recall():>10.4f} {m.f1():>10.4f}")


print("\n=== Why Linear Regression Fails for Classification ===")
print("Fitting linear regression to binary labels:")
x_hours = list(range(1, 11))
y_pass = [0, 0, 0, 0, 1, 1, 1, 1, 1, 1]

n = len(x_hours)
x_mean = sum(x_hours) / n
y_mean = sum(y_pass) / n
numerator = sum((x_hours[i] - x_mean) * (y_pass[i] - y_mean) for i in range(n))
denominator = sum((x_hours[i] - x_mean) ** 2 for i in range(n))
w_lin = numerator / denominator
b_lin = y_mean - w_lin * x_mean

print(f"\nLinear fit: y = {w_lin:.4f}*x + {b_lin:.4f}")
print(f"{'Hours':>6} {'Actual':>8} {'Linear':>8} {'Sigmoid':>8}")
for h, actual in zip(x_hours, y_pass):
    lin_pred = w_lin * h + b_lin
    sig_pred = sigmoid(3 * (h - 4.5))
    print(f"{h:>6d} {actual:>8d} {lin_pred:>8.3f} {sig_pred:>8.3f}")

print("\nLinear regression gives values outside [0, 1].")
print("Logistic regression keeps everything in [0, 1] as probabilities.")


=== Multi-class Softmax Regression (3 classes) ===
  Epoch    0 | Loss: 1.0986
  Epoch  200 | Loss: 0.1781
  Epoch  400 | Loss: 0.1115
  Epoch  600 | Loss: 0.0852
  Epoch  800 | Loss: 0.0709

Train accuracy: 0.9917
Test accuracy:  1.0000

Sample predictions:
  True: 2, Predicted: 2, Probs: [0.029, 0.000, 0.971]
  True: 2, Predicted: 2, Probs: [0.007, 0.022, 0.971]
  True: 0, Predicted: 0, Probs: [0.963, 0.003, 0.034]
  True: 2, Predicted: 2, Probs: [0.012, 0.001, 0.987]
  True: 2, Predicted: 2, Probs: [0.018, 0.000, 0.981]

=== Threshold Tuning ===
Default threshold: 0.5. Adjusting trades precision for recall.

 Threshold   Accuracy  Precision     Recall         F1
----------------------------------------------------
       0.3     0.9500     0.9130     1.0000     0.9545
       0.4     1.0000     1.0000     1.0000     1.0000
       0.5     1.0000     1.0000     1.0000     1.0000
       0.6     0.9750     1.0000     0.9524     0.9756
       0.7     0.9500     1.0000     0.9048     0.95

Scikit comparison

In [6]:
print("\n=== Scikit-learn Comparison ===")
try:
    from sklearn.linear_model import LogisticRegression as SklearnLR
    from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
    from sklearn.metrics import confusion_matrix, classification_report
    from sklearn.model_selection import train_test_split
    from sklearn.preprocessing import StandardScaler
    import numpy as np

    np.random.seed(42)
    X_0 = np.random.randn(100, 2) + [2, 2]
    X_1 = np.random.randn(100, 2) + [5, 5]
    X_sk = np.vstack([X_0, X_1])
    y_sk = np.array([0] * 100 + [1] * 100)

    X_tr, X_te, y_tr, y_te = train_test_split(X_sk, y_sk, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_tr_sc = scaler.fit_transform(X_tr)
    X_te_sc = scaler.transform(X_te)

    lr = SklearnLR()
    lr.fit(X_tr_sc, y_tr)
    y_pred_sk = lr.predict(X_te_sc)

    print(f"Accuracy:  {accuracy_score(y_te, y_pred_sk):.4f}")
    print(f"Precision: {precision_score(y_te, y_pred_sk):.4f}")
    print(f"Recall:    {recall_score(y_te, y_pred_sk):.4f}")
    print(f"F1:        {f1_score(y_te, y_pred_sk):.4f}")
    print(f"\nConfusion Matrix:\n{confusion_matrix(y_te, y_pred_sk)}")
    print(f"\nClassification Report:\n{classification_report(y_te, y_pred_sk)}")

except ImportError:
    print("scikit-learn not installed. Install with: pip install scikit-learn")
    print("The from-scratch implementations above work without any dependencies.")


=== Scikit-learn Comparison ===
Accuracy:  1.0000
Precision: 1.0000
Recall:    1.0000
F1:        1.0000

Confusion Matrix:
[[21  0]
 [ 0 19]]

Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        21
           1       1.00      1.00      1.00        19

    accuracy                           1.00        40
   macro avg       1.00      1.00      1.00        40
weighted avg       1.00      1.00      1.00        40

